<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Agent/blob/main/07%20Deep%20Agent%20Workflow/2%20Alpinion_Workflow_Dashboard_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Product-A 인허가 Workflow Dashboard
알피니언메디칼시스템 PBL · Google Colab 실습

**실습 목표**: 준비 건수, 제출 차단 사유, 문서 개정본과 시험 근거, 기관 보완 대응표를 한 화면에서 확인합니다.

1. 이 파일을 Colab에 업로드하고 **런타임 → 모두 실행**을 선택합니다.
2. 마지막 셀의 대시보드에서 **가상 승인 반영**을 켜고 끕니다.
3. **제출 준비 및 차단**에서 부서를 선택하고, **문서·시험 근거**에서 QVR-501을 확인합니다.
4. **기관 보완 대응**에서 요청과 원문 확인 항목을 비교합니다.

API 키 없이 전체 대시보드를 사용합니다. 모든 업무 기록과 승인 변경은 가상입니다. 선택 사항인 실제 Deep Agent 설명은 3절에서 활성화합니다.

## 1. 실습 데이터 준비
이전 PBL의 Product-A 카탈로그와 필수 문서 8건을 내장했습니다. 폴더와 모의 API에서 같은 개정본을 발견하면 두 출처를 보존합니다.

In [1]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from threading import Thread
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd
from IPython.display import display

PROJECT, PRODUCT, MARKET = "PA-001", "Product-A", "시장 A"
workflow = pd.DataFrame([
    ("A", "개발", "설계·시험 산출물 작성", "—"),
    ("B", "개발·RA", "제출 대상 판별 및 지정", "A"),
    ("C1", "품질", "품질·인증 시험 근거 준비", "B"),
    ("C2", "개발", "임상평가 자료 준비", "B"),
    ("C3", "구매·제조", "공급업체·제조 자료 준비", "B"),
    ("D", "RA", "STED 취합 및 완료", "C1, C2, C3"),
    ("E", "RA·승인권자", "결재 후 기관 제출", "D"),
    ("F", "RA·관련 부서", "기관 요청 시 보완 및 재제출", "E, 기관 보완 요청"),
], columns=["단계", "담당", "활동", "선행 조건"])
required = pd.DataFrame([
    ("DHF-001", "시스템 설계명세", "개발", "설계", "B"),
    ("QVR-501", "거리 측정 품질검증 보고서", "품질", "시험·검증", "D"),
    ("CER-001", "임상평가 보고서", "개발", "임상평가", "D"),
    ("SUP-001", "협력업체 승인원", "구매", "제조·공급망", "D"),
    ("DMR-001", "제조 기준 문서", "제조", "제조·공급망", "D"),
    ("STED-001", "기술문서 제출본", "RA", "인허가 제출·보완", "E"),
    ("TEST-001", "인증시험 성적서", "품질", "시험·검증", "E"),
    ("QVR-102", "영상 기능 품질검증 보고서", "품질", "시험·검증", "D"),
], columns=["document_id", "필수 문서", "책임부서", "예상 분류", "영향 단계"])
display(workflow, required)

,단계,담당,활동,선행 조건
0,A,개발,설계·시험 산출물 작성,—
1,B,개발·RA,제출 대상 판별 및 지정,A
2,C1,품질,품질·인증 시험 근거 준비,B
3,C2,개발,임상평가 자료 준비,B
4,C3,구매·제조,공급업체·제조 자료 준비,B
5,D,RA,STED 취합 및 완료,"C1, C2, C3"
6,E,RA·승인권자,결재 후 기관 제출,D
7,F,RA·관련 부서,기관 요청 시 보완 및 재제출,"E, 기관 보완 요청"


,document_id,필수 문서,책임부서,예상 분류,영향 단계
0,DHF-001,시스템 설계명세,개발,설계,B
1,QVR-501,거리 측정 품질검증 보고서,품질,시험·검증,D
2,CER-001,임상평가 보고서,개발,임상평가,D
3,SUP-001,협력업체 승인원,구매,제조·공급망,D
4,DMR-001,제조 기준 문서,제조,제조·공급망,D
5,STED-001,기술문서 제출본,RA,인허가 제출·보완,E
6,TEST-001,인증시험 성적서,품질,시험·검증,E
7,QVR-102,영상 기능 품질검증 보고서,품질,시험·검증,D


In [2]:
def record(doc_id, revision, title, kind, approval, designated=False,
           access="접근 가능", related=()):
    return dict(document_id=doc_id, revision=revision, title=title,
                source_type=kind, approval_status=approval,
                submission_status="지정" if designated else "미지정",
                access_status=access, product_id=PRODUCT, project_id=PROJECT,
                target_market=MARKET, regulation_refs=[],
                regulation_status="판본 확인 필요", related_ids=list(related))

folder_rows = [
    record("DHF-001", "1.0", "시스템 설계명세", "DESIGN", "승인", True),
    record("QVR-501", "1.0", "거리 측정 품질검증 보고서", "QVR", "승인",
           related=("EX-SRS-501", "EX-QVS-501")),
    record("QVR-501", "1.1", "거리 측정 품질검증 개정본", "QVR", "검토 중", True,
           related=("EX-SRS-501", "EX-QVS-501")),
    record("CER-001", "1.0", "임상평가 보고서", "CER", "승인", True),
    record("CER-001", "1.1", "임상평가 개정 초안", "CER", "검토 중"),
    record("STED-001", "0.2", "STED 취합 초안", "STED", "검토 중", True),
]
api_seed = [
    folder_rows[1].copy(),
    record("DMR-001", "1.0", "제조 기준 문서", "DMR", "승인", True, "접근 제한"),
    record("TEST-001", "2.0", "인증시험 성적서", "TEST", "승인"),
    record("QVR-102", "1.9", "영상 기능 품질검증 보고서", "QVR", "승인", True),
    record("QVR-102", "1.10", "영상 기능 품질검증 개정본", "QVR", "검토 중"),
    record("CR-001", "1.0", "거리 측정 시험 조건 변경 기록", "CHANGE", "승인",
           related=("EX-SRS-501",)),
]

def source_key(row):
    return f"{row['document_id']}@{row['revision']}"

with TemporaryDirectory() as workspace:
    project_folder = Path(workspace) / "Product-A"
    project_folder.mkdir()
    for row in folder_rows:
        file = project_folder / f"{source_key(row)}.json"
        file.write_text(json.dumps(row, ensure_ascii=False), encoding="utf-8")
    from_folder = []
    for file in sorted(project_folder.glob("*.json")):
        row = json.loads(file.read_text(encoding="utf-8"))
        from_folder.append(dict(row, source_id=file.stem, source_ref=f"폴더 Product-A/{file.name}"))

class CatalogHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path != "/documents":
            self.send_error(404)
            return
        body = json.dumps({"documents": api_seed}, ensure_ascii=False).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *_):
        pass

with ThreadingHTTPServer(("127.0.0.1", 0), CatalogHandler) as server:
    thread = Thread(target=server.serve_forever, daemon=True)
    thread.start()
    try:
        with urlopen(f"http://127.0.0.1:{server.server_port}/documents", timeout=10) as response:
            api_rows = json.load(response)["documents"]
    finally:
        server.shutdown()
        thread.join()

from_api = [
    dict(row, source_id=source_key(row), source_ref=f"모의 API /documents · {source_key(row)}")
    for row in api_rows
]
raw = pd.DataFrame(from_folder + from_api)
display(raw[["source_id", "source_ref", "approval_status", "access_status"]])

taxonomy = {
    "DESIGN": "설계", "CHANGE": "설계", "QVR": "시험·검증",
    "TEST": "시험·검증", "CER": "임상평가", "SUP": "제조·공급망",
    "DMR": "제조·공급망", "STED": "인허가 제출·보완",
}
schema = pd.DataFrame([
    ("document_id + revision", "문자열 복합키", "이 예시의 개정본 식별"),
    ("product_id + project_id + target_market", "문자열", "제출 프로젝트의 제품·시장 범위"),
    ("document_type", "허용값", "Taxonomy에서 매핑"),
    ("regulation_refs", "배열", "확인한 규정과 판본 목록"),
    ("regulation_status", "상태", "판본 확인 여부"),
    ("approval_status", "허용값", "문서 승인 또는 검토 중"),
    ("submission_status", "허용값", "프로젝트별 지정 또는 미지정"),
    ("access_status", "허용값", "원본 접근 가능 또는 접근 제한"),
    ("source_id + source_ref", "문자열", "소스 내 식별자와 위치"),
    ("related_ids", "배열", "관련 요구사항·시험 식별자"),
], columns=["필드", "유형", "규칙"])
raw["document_type"] = raw["source_type"].map(taxonomy).fillna("분류 확인 필요")
assert raw["revision"].str.fullmatch(r"[0-9]+[.][0-9]+").all()
assert not raw["document_type"].eq("분류 확인 필요").any()
display(schema)

scope = ["product_id", "project_id", "target_market"]
key = scope + ["document_id", "revision"]
tracked = ["approval_status", "submission_status", "access_status", "document_type"]
catalog_rows = []
for _, group in raw.groupby(key, sort=False):
    item = group.iloc[0].to_dict()
    item["source_ref"] = " / ".join(sorted(set(group["source_ref"])))
    item["source_count"] = group["source_ref"].nunique()
    item["conflict"] = any(group[field].nunique() > 1 for field in tracked)
    item["revision_order"] = tuple(map(int, item["revision"].split(".")))
    catalog_rows.append(item)
catalog = pd.DataFrame(catalog_rows)
assert len(raw) == 12 and len(catalog) == 11
display(catalog[["document_id", "revision", "document_type", "approval_status",
                 "submission_status", "source_count", "source_ref"]])

,source_id,source_ref,approval_status,access_status
0,CER-001@1.0,폴더 Product-A/CER-001@1.0.json,승인,접근 가능
1,CER-001@1.1,폴더 Product-A/CER-001@1.1.json,검토 중,접근 가능
2,DHF-001@1.0,폴더 Product-A/DHF-001@1.0.json,승인,접근 가능
3,QVR-501@1.0,폴더 Product-A/QVR-501@1.0.json,승인,접근 가능
4,QVR-501@1.1,폴더 Product-A/QVR-501@1.1.json,검토 중,접근 가능
5,STED-001@0.2,폴더 Product-A/STED-001@0.2.json,검토 중,접근 가능
6,QVR-501@1.0,모의 API /documents · QVR-501@1.0,승인,접근 가능
7,DMR-001@1.0,모의 API /documents · DMR-001@1.0,승인,접근 제한
8,TEST-001@2.0,모의 API /documents · TEST-001@2.0,승인,접근 가능
9,QVR-102@1.9,모의 API /documents · QVR-102@1.9,승인,접근 가능


,필드,유형,규칙
0,document_id + revision,문자열 복합키,이 예시의 개정본 식별
1,product_id + project_id + target_market,문자열,제출 프로젝트의 제품·시장 범위
2,document_type,허용값,Taxonomy에서 매핑
3,regulation_refs,배열,확인한 규정과 판본 목록
4,regulation_status,상태,판본 확인 여부
5,approval_status,허용값,문서 승인 또는 검토 중
6,submission_status,허용값,프로젝트별 지정 또는 미지정
7,access_status,허용값,원본 접근 가능 또는 접근 제한
8,source_id + source_ref,문자열,소스 내 식별자와 위치
9,related_ids,배열,관련 요구사항·시험 식별자


,document_id,revision,document_type,approval_status,submission_status,source_count,source_ref
0,CER-001,1.0,임상평가,승인,지정,1,폴더 Product-A/CER-001@1.0.json
1,CER-001,1.1,임상평가,검토 중,미지정,1,폴더 Product-A/CER-001@1.1.json
2,DHF-001,1.0,설계,승인,지정,1,폴더 Product-A/DHF-001@1.0.json
3,QVR-501,1.0,시험·검증,승인,미지정,2,모의 API /documents · QVR-501@1.0 / 폴더 Product-A...
4,QVR-501,1.1,시험·검증,검토 중,지정,1,폴더 Product-A/QVR-501@1.1.json
5,STED-001,0.2,인허가 제출·보완,검토 중,지정,1,폴더 Product-A/STED-001@0.2.json
6,DMR-001,1.0,제조·공급망,승인,지정,1,모의 API /documents · DMR-001@1.0
7,TEST-001,2.0,시험·검증,승인,미지정,1,모의 API /documents · TEST-001@2.0
8,QVR-102,1.9,시험·검증,승인,지정,1,모의 API /documents · QVR-102@1.9
9,QVR-102,1.10,시험·검증,검토 중,미지정,1,모의 API /documents · QVR-102@1.10


In [3]:
def summarize_versions(group):
    ordered = group.sort_values("revision_order", ascending=False)
    approved = ordered[ordered["approval_status"].eq("승인") & ~ordered["conflict"]]
    selected = ordered[ordered["submission_status"].eq("지정")]
    return {
        "최신 작성본": ordered.iloc[0]["revision"],
        "최신 승인본": approved.iloc[0]["revision"] if len(approved) else "—",
        "제출 지정본": ", ".join(selected["revision"]) if len(selected) else "—",
    }

version_table = pd.DataFrame([
    dict(document_id=doc_id, **summarize_versions(group))
    for doc_id, group in catalog.groupby("document_id")
])
display(version_table)

,document_id,최신 작성본,최신 승인본,제출 지정본
0,CER-001,1.1,1.0,1.0
1,CR-001,1.0,1.0,—
2,DHF-001,1.0,1.0,1.0
3,DMR-001,1.0,1.0,1.0
4,QVR-102,1.10,1.9,1.9
5,QVR-501,1.1,1.0,1.1
6,STED-001,0.2,—,0.2
7,TEST-001,2.0,2.0,—


## 2. 제출 준비 계산과 문서 상세
대시보드는 기존 검증 규칙으로 상태를 계산합니다. QVR-501만 가상 승인해도 다른 항목이 남으면 계속 자료 준비 단계입니다. 화면 표기의 RA 검토 가능은 실습 체크리스트 기준입니다.

In [4]:
def assess(group, expected_type):
    if group.empty:
        return "누락", "원본 위치와 확보 일정을 확인"
    if group["conflict"].any():
        return "출처 간 충돌", "원본 시스템에서 메타데이터를 대조"
    designated = group[group["submission_status"].eq("지정")]
    if len(designated) != 1:
        return "제출 지정 확인", "RA와 제출 지정본을 하나로 확정"
    row = designated.iloc[0]
    if row["document_type"] != expected_type:
        return "유형 확인", "필수 목록과 문서 분류를 대조"
    if row["access_status"] != "접근 가능":
        return "접근 제한", "권한 또는 승인된 공유 경로를 확인"
    if row["approval_status"] != "승인":
        return "승인 대기", "지정 개정본의 검토 및 승인 근거를 확인"
    return "준비", "제출 검토에 전달"


def calculate_dashboard(approved=False):
    working = catalog.copy()
    if approved:
        revised = working["document_id"].eq("QVR-501") & working["revision"].eq("1.1")
        working.loc[revised, "approval_status"] = "승인"


    scoped = working[(working[scope] == [PRODUCT, PROJECT, MARKET]).all(axis=1)]
    results = []
    for requirement in required.to_dict("records"):
        docs = scoped[scoped["document_id"].eq(requirement["document_id"])]
        status, action = assess(docs, requirement["예상 분류"])
        versions = summarize_versions(docs) if len(docs) else dict.fromkeys(
            ["최신 작성본", "최신 승인본", "제출 지정본"], "—"
        )
        results.append(dict(requirement, **versions, 상태=status, 다음_조치=action,
                            관찰_근거=" / ".join(sorted(set(docs["source_ref"]))) if len(docs) else "두 소스 목록에서 미확인"))
    readiness = pd.DataFrame(results)
    bottlenecks = readiness[readiness["상태"].ne("준비")].copy()
    return scoped, readiness, bottlenecks

In [5]:
from datetime import datetime
from html import escape
from zoneinfo import ZoneInfo

TRACE = {"QVR-501": ("EX-SRS-501", "EX-QVS-501", "교육용 Traceability Matrix")}
VERSION_COLUMNS = ["revision", "approval_status", "submission_status",
                   "access_status", "source_ref"]
TABLE_COLUMNS = ["document_id", "필수 문서", "제출 지정본", "상태",
                 "책임부서", "영향 단계", "다음_조치"]

def document_detail(scoped, readiness, document_id):
    decision = readiness[readiness["document_id"].eq(document_id)]
    revisions = scoped[scoped["document_id"].eq(document_id)][VERSION_COLUMNS]
    return decision, revisions, TRACE.get(document_id)

def supplement_table(readiness):
    item = readiness.set_index("document_id").loc["QVR-501"]
    req, plan, _ = TRACE["QVR-501"]
    return pd.DataFrame([{
        "요청 ID": "SIM-SUP-001",
        "기관 요청": "거리 측정 시험 조건과 결과를 보완해 주세요.",
        "요구사항·시험계획": f"{req} / {plan}",
        "대응 문서": f"QVR-501 v{item['제출 지정본']}",
        "문서 상태": item["상태"],
        "담당": item["책임부서"],
        "추가 확인": "시험 조건과 실측 결과는 원문 확인 필요",
        "RA 검토": "검토 대기",
    }])

# 화면 연결 전에 두 시나리오를 검증합니다.
before = calculate_dashboard(False)
after = calculate_dashboard(True)
assert len(before[1]) == len(after[1]) == 8
assert len(before[2]) == 5 and len(after[2]) == 4
assert "QVR-501" in set(before[2]["document_id"])
assert "QVR-501" not in set(after[2]["document_id"])
assert after[1].set_index("document_id").loc["QVR-501", "최신 승인본"] == "1.1"
print("검증 완료 · 3/8건 → 4/8건 · 두 경우 모두 차단 항목이 남습니다.")

검증 완료 · 3/8건 → 4/8건 · 두 경우 모두 차단 항목이 남습니다.


## 3. Deep Agent 연결 선택 사항
기본값으로 두면 모델 연결 없이 대시보드가 동작합니다. **실제 Agent 설명**을 보려면 Colab 보안 비밀에 `OPENAI_API_KEY`를 등록하고 액세스를 켠 뒤 `ENABLE_AGENT`를 체크합니다. 이후 대시보드 셀을 다시 실행하고 문서 상세에서 **Agent로 설명**을 누릅니다. 버튼을 누를 때 모델 API를 호출합니다.

Agent는 선택한 상태와 문서 개정본을 읽고 설명을 작성합니다. 대시보드의 준비 판정은 Python 규칙으로 계산됩니다.

[LangChain Deep Agents Quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart)

In [ ]:
import os
import getpass

# 사용자로부터 OpenAI API Key를 안전하게 입력받습니다.
api_key = getpass.getpass("OpenAI API Key를 입력하세요: ")

if api_key:
    os.environ["OPENAI_API_KEY"] = api_key
    print("\nSUCCESS: OpenAI API Key가 환경 변수('OPENAI_API_KEY')에 성공적으로 등록되었습니다.")
else:
    print("\nWARNING: API Key가 입력되지 않았습니다.")

In [7]:
ENABLE_AGENT = True  # @param {type:"boolean"}
MODEL = "openai:gpt-6.1-sol"  # @param {type:"string"}

if ENABLE_AGENT:
    import os
    import subprocess
    import sys
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "deepagents", "langchain-openai"
    ])
    from deepagents import create_deep_agent

def explain_with_agent(scoped, readiness, document_id):
    decision, revisions, trace = document_detail(scoped, readiness, document_id)

    def read_selected_document() -> dict:
        '''선택된 문서의 검증 결과, 개정본과 출처, 시험 근거를 조회합니다.'''
        return {
            "document": document_id,
            "decision": decision.to_dict("records"),
            "versions": revisions.to_dict("records"),
            "trace": trace,
        }

    agent = create_deep_agent(
        model=MODEL, tools=[read_selected_document],
        system_prompt=(
            "가상 Product-A의 RA 업무 보조자입니다. read_selected_document를 먼저 호출하세요. "
            "판정 결과, 문서 개정본, 출처, 담당부서와 다음 조치를 그대로 근거로 사용하세요. "
            "시험 원문의 수치가 제공되지 않으면 원문 확인 필요로 표시하세요."
        ),
    )
    result = agent.invoke({
        "messages": [{"role": "user", "content": f"{document_id}의 상태와 다음 조치를 설명해 주세요."}]
    })
    tool_names = [
        call["name"] for msg in result["messages"]
        for call in getattr(msg, "tool_calls", [])
    ]
    if "read_selected_document" not in tool_names:
        return "문서 조회 호출을 확인하지 못했습니다. 상세 표에서 근거를 확인하세요."
    return "도구 호출 · " + ", ".join(tool_names) + "\n\n" + str(result["messages"][-1].content)

## 4. 대시보드 실행
- **프로젝트 현황** · 준비 건수와 현재 단계
- **제출 준비 및 차단** · 부서 필터와 전체 필수 문서 보기
- **문서·시험 근거** · 개정본, 원본 위치, 검증 결과 및 연결된 시험 근거
- **기관 보완 대응** · 가상 요청과 RA 검토용 대응표

**실습 확인**: 가상 승인 전후로 준비 문서가 3건에서 4건으로 바뀌는지 확인하세요. QVR-501은 차단 목록에서 사라지고, 나머지 4건은 남습니다.

In [9]:
try:
    import ipywidgets as widgets
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as widgets

from IPython.display import HTML, clear_output

approval = widgets.Checkbox(value=False, description="가상 승인 반영")
department = widgets.Dropdown(
    options=["전체", *sorted(required["책임부서"].unique())], description="담당부서"
)
blocked_only = widgets.Checkbox(value=True, description="차단 항목만")
document = widgets.Dropdown(
    options=[(f"{row['document_id']} · {row['필수 문서']}", row["document_id"])
             for row in required.to_dict("records")],
    value="QVR-501", description="문서", layout=widgets.Layout(width="min(100%, 510px)")
)
agent_button = widgets.Button(description="Agent로 설명", disabled=not ENABLE_AGENT)
summary = widgets.HTML()
queue = widgets.HTML()
detail = widgets.HTML()
supplement = widgets.HTML()
agent_output = widgets.Output()

def as_table(frame):
    if frame.empty:
        return "<p>현재 조건에 해당하는 항목이 없습니다.</p>"
    return '<div style="overflow-x:auto">' + frame.to_html(index=False, escape=True) + "</div>"

def selected_snapshot():
    return calculate_dashboard(approval.value)

def refresh(_=None):
    scoped, readiness, bottlenecks = selected_snapshot()
    count = len(readiness) - len(bottlenecks)
    stage = "자료 준비 · 제출 조건 미충족" if len(bottlenecks) else "RA 제출본 검토 가능"
    timestamp = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y-%m-%d %H:%M KST")
    summary.value = (
        f"<div style='padding:16px;border-left:5px solid #155e75'>"
        f"<h2 style='margin:0 0 8px'>{escape(PRODUCT)} 인허가 현황</h2>"
        f"<p>{escape(PROJECT)} · {escape(MARKET)} · 실습 체크리스트 v1.0</p>"
        f"<p><b>준비 문서 {count}/{len(readiness)}건</b> · 차단 {len(bottlenecks)}건</p>"
        f"<p>{stage} · {timestamp}</p></div>"
    )
    rows = bottlenecks if blocked_only.value else readiness
    if department.value != "전체":
        rows = rows[rows["책임부서"].eq(department.value)]
    queue.value = as_table(rows[TABLE_COLUMNS])

    decision, revisions, trace = document_detail(scoped, readiness, document.value)
    detail.value = (
        "<h3>준비 판정과 개정본</h3>"
        + as_table(decision[TABLE_COLUMNS + ["최신 작성본", "최신 승인본", "관찰_근거"]])
        + "<h3>개정본별 원본 위치</h3>" + as_table(revisions)
        + ("<p>시험 근거 · " + escape(" / ".join(trace)) + "</p>" if trace
           else "<p>실습 추적표에 연결된 시험 근거가 없습니다.</p>")
    )
    supplement.value = as_table(supplement_table(readiness))

def reset_explanation(change):
    agent_output.clear_output()
    refresh(change)

def on_explain(_):
    agent_button.disabled = True
    with agent_output:
        clear_output(wait=True)
        print("선택 문서의 근거를 조회 중입니다.")
        try:
            scoped, readiness, _ = selected_snapshot()
            answer = explain_with_agent(scoped, readiness, document.value)
            clear_output(wait=True)
            print(answer)
        except Exception as error:
            clear_output(wait=True)
            print(f"Agent 호출을 완료하지 못했습니다 ({type(error).__name__}). API 키, 모델 접근 권한, 연결 상태를 확인하세요.")
        finally:
            agent_button.disabled = False

approval.observe(reset_explanation, names="value")
document.observe(reset_explanation, names="value")
department.observe(refresh, names="value")
blocked_only.observe(refresh, names="value")
agent_button.on_click(on_explain)

tabs = widgets.Tab(children=[
    summary, widgets.VBox([department, blocked_only, queue]),
    widgets.VBox([document, detail, agent_button, agent_output]), supplement,
])
for idx, title in enumerate(["프로젝트 현황", "제출 준비 및 차단", "문서·시험 근거", "기관 보완 대응"]):
    tabs.set_title(idx, title)
refresh()
display(widgets.VBox([approval, tabs]))